# Автоматическая обработка данных такси


- Автор: Шабарина Д.Е.


## Описание проекта

 Каждый день компания обрабатывает миллионы поездок, оплаченных разными способами. Чтобы финансовая и продуктовая команды регулярно получали актуальные данные о выручке и поведении пассажиров, коллеги решили настроить автоматическую обработку этих данных.

Задача — построить витрину данных, которая будет агрегировать информацию о поездках по каждому способу оплаты. Для этого нужно напишем PySpark-скрипт, который рассчитает ключевые показатели: количество поездок, среднюю стоимость поездки, средние чаевые и суммарную выручку по каждому типу оплаты.

Чтобы процесс был полностью автоматическим и не зависел от ручных запусков, создадим DAG в Airflow. Этот DAG должен ежедневно:

* проверять наличие новых файлов с данными;

* запускать Spark-задачу;

* формировать обновлённую итоговую таблицу.

Эта таблица станет основой для финансовых отчётов и аналитических дашбордов.

## Описание данных

Таблица `taxi_data` содержит данные об активности пользователей и состоит из следующих полей:

* `taxi_id` — идентификатор водителя;

* `trip_start_timestamp` — время начала поездки;

* `trip_end_timestamp` — время окончания поездки;

* `trip_seconds` — длительность поездки в секундах;

* `trip_miles` — дистанция поездки;

* `fare` — стоимость поездки;

* `tips` — размер чаевых;

* `trip_total` — общая стоимость поездки: стоимость поездки + чаевые + комиссия;

* `payment_type` — способ оплаты.

## Что нужно сделать

- автоматизировать подготовку витрины данных по поездкам Такси:

1. Сначала пишем Spark-скрипт, который будет обрабатывать данные о поездках и агрегировать показатели по способам оплаты `payment_type`. Для этого рассчитаем несколько показателей:

* количество поездок, которое показывает общий спрос и загрузку сервиса;
* среднюю стоимость `fare`, которое отражает уровень среднего чека поездки;
* средние чаевые `tips` — индикатор удовлетворённости клиентов и мотивации водителей;
* суммарную выручку `trip_total` — ключевой показатель дохода компании.

Все результаты будут собираться в одну итоговую таблицу `taxi_payment_summary`. После этого таблицу запишем в ClickHouse с помощью JDBC-драйвера.

2. Далее настроим DAG в Airflow. DAG должен запускаться ежедневно. Перед запуском он проверяет наличие файла с данными за нужную дату в S3-хранилище и только после появления файла запускает Spark-задачу.


## Шаг 1. Настройте Spark-агрегацию

Данные хранятся в формате Parquet, поэтому для чтения используем метод `spark.read.parquet()`. Это быстрее и надёжнее, чем CSV.

Сгруппируем данные по полю `payment_type` и рассчитаем четыре показателя:

* количество поездок — `count(*)`;
* среднюю стоимость — `avg(...)`;
* средние чаевые — `avg(...)`;
* суммарную выручку — `sum(...)`.

Так получится витрина для анализа информации по каждому способу оплаты. После этого настроим запись полученной таблицы в ClickHouse. 

In [ ]:
# filename=my_spark_job.py
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import sys

# Создаём Spark-сессию и при необходимости добавляем конфигурации
spark = SparkSession.builder \
    .appName("myAggregateTest") \
    .config("fs.s3a.endpoint", "storage.yandexcloud.net") \
    .getOrCreate()

# Указываем параметры для подключения к ClickHouse
jdbcPort = 8443
jdbcHostname = "rc1a-3jouval14nne7aun.mdb.yandexcloud.net"
username = "da_20260106_ea7ec52eee"
jdbcDatabase = "playground_" + username
jdbcUrl = f"jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true"

# Считываем исходные данные за нужную дату
taxiData = spark.read.parquet(f"s3a://da-plus-dags/project_04/taxi_data.parquet")

# Выполняем агрегацию по payment_type
result_df = taxiData.groupBy("payment_type").agg(
    F.count("*").alias("trip_count"),
    F.avg("fare").alias("avg_fare"),
    F.avg("tips").alias("avg_tips"),
    F.sum("trip_total").alias("total_revenue")
)

# Записываем результат в ClickHouse
result_df.write.format("jdbc") \
    .option("url", jdbcUrl) \
    .option("user", username) \
    .option("password", "f81df282a24d4202bc0aa47c6cfd5326") \
    .option("dbtable", "taxi_payment_summary") \
    .mode('append') \
    .save()

## Шаг 2. Настройте DAG

В DAG используем `S3KeySensor`, чтобы дождаться появления файла в S3. После этого запускаем `DataprocCreatePysparkJobOperator`, передав путь к нашему скрипту. 

Ваши данные для подключения к Airflow:
*   IP — 89.169.152.149
*   Имя пользователя — da_20260106_ea7ec52eee
*   Пароль — f81df282a24d4202bc0aa47c6cfd5326

In [ ]:
# filename=taxi_payment_summary_dag.py
from datetime import datetime, timedelta
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

class PysparkJobOperator(DataprocCreatePysparkJobOperator):
    template_fields = ("cluster_id",)

DAG_ID = "taxi_payment_summary_daily"

with DAG(
    DAG_ID,
    schedule_interval='@daily',
    start_date=datetime(2025, 1, 1),  
    catchup=False
) as dag:
    # Сенсор, ждем появления файла в S3
    wait_for_taxi_data = S3KeySensor(
        task_id='wait_for_taxi_data',
        bucket_name='da-plus-dags',
        bucket_key='project_04/taxi_data.parquet',
        poke_interval=300,  # 5 минут
        timeout=60 * 60,     # 1 час
        mode='poke',
        aws_conn_id='s3',
        soft_fail=False
    )

    # Запуск PySpark задания через Dataproc
    run_spark_job = DataprocCreatePysparkJobOperator(
        task_id='run_spark_job',
        cluster_id='c9q4134h5vi546h1e148',
        name='taxi_payment_summary',
        main_python_file_uri=f"s3a://da-plus-dags/da_20260106_ea7ec52eee/jobs/my_spark_job.py"
    )

    wait_for_taxi_data >> run_spark_job

## Шаг 3. Запуск DAG с помощью Airflow UI



Ссылка на скриншоты запуска: https://disk.yandex.ru/d/fcnlyVAhCz0HbA